<a href="https://colab.research.google.com/github/snumryk/TRPA1-ML-benchmark/blob/main/scripts/GroupKFold_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Встановлення

In [ ]:
# One version to rule them all: 4.44.2 works with MolFormer's old revision
!pip install "transformers==4.44.2" "tokenizers<0.20" rdkit xgboost scipy scikit-learn -q
print("Installed. NOW: Runtime → Restart session, then run Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 24.8 MB/s eta 0:00:00
Installed. NOW: Runtime → Restart session, then run Cell 2.


## Mount + перевірка версій

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_PATH = '/content/drive/MyDrive/trpa1_project'
print("Files:", os.listdir(DATA_PATH))

import transformers, rdkit, xgboost
print(f"transformers={transformers.__version__} (must be 4.44.2)")
print(f"rdkit={rdkit.__version__}, xgboost={xgboost.__version__}")

Mounted at /content/drive
Files: ['trpa1_antagonists.csv', 'decoys_clean.csv', 'embeddings_all.npz', 'cv_results.json']
transformers=4.44.2 (must be 4.44.2)
rdkit=2026.03.3, xgboost=3.3.0


## Витягування ВСІХ embeddings (ChemBERTa + MolFormer)

In [ ]:
"""
Extract ChemBERTa (CLS) + MolFormer (MeanPool) embeddings for all 1645 molecules.
MolFormer pinned to old revision 7b12d946 (compatible with transformers 4.44.2).
"""
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm import tqdm

DATA_PATH = '/content/drive/MyDrive/trpa1_project'
df = pd.read_csv(f'{DATA_PATH}/trpa1_antagonists.csv')
print(f"Molecules: {len(df)}, scaffolds: {df['scaffold'].nunique()}")

# ── RDKit descriptors ──────────────────────────────────────────
RDKIT_DESCS = [
    'MolWt', 'MolLogP', 'MolMR', 'TPSA',
    'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds',
    'NumAromaticRings', 'RingCount', 'FractionCSP3',
    'HeavyAtomCount', 'NumAliphaticRings', 'NumSaturatedRings',
    'NumHeteroatoms', 'LabuteASA',
]

def compute_rdkit(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(len(RDKIT_DESCS))
    return np.array([float(getattr(Descriptors, n)(mol)) for n in RDKIT_DESCS], dtype=np.float32)

X_rdkit_all = np.vstack(df['std_smiles'].apply(compute_rdkit).values)
print(f"RDKit: {X_rdkit_all.shape}")

smiles_all = df['std_smiles'].tolist()

def extract_all(smiles_list, model, tokenizer, pooling, max_length):
    embeddings = []
    for smi in tqdm(smiles_list, desc=pooling):
        tokens = tokenizer(smi, return_tensors="pt", truncation=True,
                           padding=True, max_length=max_length).to('cuda')
        with torch.no_grad():
            output = model(**tokens)
        hidden = output.last_hidden_state
        if pooling == 'cls':
            emb = hidden[:, 0, :].cpu().numpy().ravel()
        else:
            mask = tokens['attention_mask'].unsqueeze(-1).float().to('cuda')
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            emb = emb.cpu().numpy().ravel()
        embeddings.append(emb)
    return np.vstack(embeddings)

# ── ChemBERTa (CLS) ────────────────────────────────────────────
print("\n[1/2] ChemBERTa...")
cb_name = "DeepChem/ChemBERTa-77M-MTR"
cb_tok = AutoTokenizer.from_pretrained(cb_name)
cb_model = AutoModel.from_pretrained(cb_name).to('cuda').eval()
X_cb_cls_all = extract_all(smiles_all, cb_model, cb_tok, 'cls', 128)
del cb_model; torch.cuda.empty_cache()
print(f"ChemBERTa CLS: {X_cb_cls_all.shape}")

# ── MolFormer (MeanPool, PINNED revision) ──────────────────────
print("\n[2/2] MolFormer...")
mf_name = "ibm/MoLFormer-XL-both-10pct"
OLD_REV = "7b12d946c181a37f6012b9dc3b002275de070314"
mf_tok = AutoTokenizer.from_pretrained(mf_name, trust_remote_code=True, revision=OLD_REV)
mf_model = AutoModel.from_pretrained(mf_name, deterministic_eval=True,
                                     trust_remote_code=True, revision=OLD_REV).to('cuda').eval()
X_mf_mean_all = extract_all(smiles_all, mf_model, mf_tok, 'mean', 202)
del mf_model; torch.cuda.empty_cache()
print(f"MolFormer MeanPool: {X_mf_mean_all.shape}")

# ── Save ───────────────────────────────────────────────────────
np.savez(f'{DATA_PATH}/embeddings_all.npz',
         rdkit=X_rdkit_all, cb_cls=X_cb_cls_all, mf_mean=X_mf_mean_all,
         y=df['pchembl_median'].values, scaffold=df['scaffold'].values)
print(f"\nSaved to embeddings_all.npz — ready for CV.")

Molecules: 1645, scaffolds: 544
RDKit: (1645, 15)

[1/2] ChemBERTa...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
cls: 100%|██████████| 1645/1645 [00:06<00:00, 261.13it/s]


ChemBERTa CLS: (1645, 384)

[2/2] MolFormer...


mean: 100%|██████████| 1645/1645 [00:34<00:00, 47.66it/s]


MolFormer MeanPool: (1645, 768)

Saved to embeddings_all.npz — ready for CV.


## GroupKFold CV

In [ ]:
"""
5-fold GroupKFold CV (groups = Murcko scaffolds) for frozen-embedding models.
Statistical backbone of the paper: mean ± std for every metric.
"""
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, matthews_corrcoef
from scipy.stats import spearmanr
import json

SEED = 42
THRESHOLD = 7.0
DATA_PATH = '/content/drive/MyDrive/trpa1_project'

data = np.load(f'{DATA_PATH}/embeddings_all.npz', allow_pickle=True)
X_rdkit, X_cb_cls, X_mf_mean = data['rdkit'], data['cb_cls'], data['mf_mean']
y, scaffolds = data['y'], data['scaffold']
print(f"Loaded: RDKit{X_rdkit.shape}, CB{X_cb_cls.shape}, MF{X_mf_mean.shape}")
print(f"{len(y)} molecules, {len(np.unique(scaffolds))} scaffolds")

def make_rf():
    return RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=SEED)
def make_xgb():
    return XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                        n_jobs=-1, random_state=SEED)

configs = {
    'RF (RDKit-15)':        (X_rdkit, make_rf),
    'XGB CB-CLS+RDKit':     (np.hstack([X_cb_cls, X_rdkit]), make_xgb),
    'XGB MF-Mean+RDKit':    (np.hstack([X_mf_mean, X_rdkit]), make_xgb),
    'RF MF-Mean+RDKit':     (np.hstack([X_mf_mean, X_rdkit]), make_rf),
    'XGB CB-CLS only':      (X_cb_cls, make_xgb),
    'XGB MF-Mean only':     (X_mf_mean, make_xgb),
}

gkf = GroupKFold(n_splits=5)

def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    rho = spearmanr(y_true, y_pred).correlation
    y_cls = (y_true >= THRESHOLD).astype(int)
    if len(np.unique(y_cls)) < 2:
        return rmse, r2, rho, np.nan, np.nan
    auc = roc_auc_score(y_cls, y_pred)
    mcc = matthews_corrcoef(y_cls, (y_pred >= THRESHOLD).astype(int))
    return rmse, r2, rho, auc, mcc

print(f"\n{'='*95}")
print("5-FOLD GroupKFold CV (groups = scaffolds) — mean ± std")
print(f"{'='*95}")
print(f"{'Model':<22} {'RMSE':>13} {'R2':>13} {'Spearman':>13} {'AUC':>13} {'MCC':>12}")
print("-"*95)

summary = {}
for name, (X, make_model) in configs.items():
    fm = {'rmse': [], 'r2': [], 'rho': [], 'auc': [], 'mcc': []}
    for tr, te in gkf.split(X, y, groups=scaffolds):
        m = make_model()
        m.fit(X[tr], y[tr])
        p = m.predict(X[te])
        rmse, r2, rho, auc, mcc = metrics(y[te], p)
        for k, v in zip(['rmse','r2','rho','auc','mcc'], [rmse,r2,rho,auc,mcc]):
            fm[k].append(v)

    def ms(vals):
        vals = np.array(vals, dtype=float)
        vals = vals[~np.isnan(vals)]
        return np.mean(vals), np.std(vals)

    res = {k: ms(fm[k]) for k in fm}
    res['r2_folds'] = fm['r2']
    summary[name] = res
    print(f"{name:<22} {res['rmse'][0]:>6.3f}±{res['rmse'][1]:.3f} "
          f"{res['r2'][0]:>6.3f}±{res['r2'][1]:.3f} "
          f"{res['rho'][0]:>6.3f}±{res['rho'][1]:.3f} "
          f"{res['auc'][0]:>6.3f}±{res['auc'][1]:.3f} "
          f"{res['mcc'][0]:>5.3f}±{res['mcc'][1]:.3f}")

print("-"*95)
best = max(summary, key=lambda k: summary[k]['r2'][0])
print(f"\nHighest mean R2: {best} ({summary[best]['r2'][0]:.3f} ± {summary[best]['r2'][1]:.3f})")
print(f"Per-fold R2: {[f'{x:.3f}' for x in summary[best]['r2_folds']]}")

# Save
save = {k: {m: list(v[m]) for m in ['r2','rmse','rho','auc','mcc']} for k, v in summary.items()}
with open(f'{DATA_PATH}/cv_results.json', 'w') as f:
    json.dump(save, f, indent=2)
print("\nSaved cv_results.json")

Loaded: RDKit(1645, 15), CB(1645, 384), MF(1645, 768)
1645 molecules, 544 scaffolds

5-FOLD GroupKFold CV (groups = scaffolds) — mean ± std
Model                           RMSE            R2      Spearman           AUC          MCC
-----------------------------------------------------------------------------------------------
RF (RDKit-15)           0.664±0.053  0.478±0.163  0.695±0.089  0.848±0.051 0.524±0.126
XGB CB-CLS+RDKit        0.634±0.057  0.536±0.097  0.728±0.073  0.869±0.049 0.551±0.120
XGB MF-Mean+RDKit       0.634±0.040  0.528±0.127  0.724±0.071  0.868±0.050 0.537±0.127
RF MF-Mean+RDKit        0.639±0.041  0.524±0.112  0.709±0.078  0.863±0.052 0.530±0.138
XGB CB-CLS only         0.633±0.057  0.539±0.084  0.725±0.064  0.866±0.043 0.559±0.113
XGB MF-Mean only        0.639±0.042  0.529±0.085  0.703±0.066  0.857±0.045 0.539±0.122
-----------------------------------------------------------------------------------------------

Highest mean R2: XGB CB-CLS only (0.539 ± 0.084)
Per-